<div style="background-color:#007033; color:white; padding:18px 22px; border-radius:6px; margin-bottom:10px;"><h1 style="color:white; margin:0 0 4px 0;">Week 2 - Readiness Assessment and Deployment Environment Planning</h1><div style="font-size:15px; opacity:0.9;">FraudGuard Case Study · Stage 2 of 10 · INFT 41000, Omar Altrad, Durham College</div></div>

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">This week</b><br>Last week you built FraudGuard and listed, informally, why it isn't production-ready. This week does two things in sequence: first, formalize that into a scored <b>readiness assessment</b> across seven dimensions, ending in an actual Go / No-Go recommendation. Then, using the Infrastructure gap that assessment almost certainly surfaces, compare on-premise, cloud, and hybrid deployment for FraudGuard using a real cost model and a weighted decision matrix, and make (and justify) an actual recommendation - the same two decisions a real ML team makes before a launch.</div>

| # | Section |
|---|---|
| 0 | Setup |
| 1 | Load Week 1's artifacts |
| 2 | The readiness framework |
| 3 | Data readiness |
| 4 | Model readiness |
| 5 | Infrastructure, organizational, risk, cost, operational readiness |
| 6 | Build the scorecard & recommendation |
| 7 | Environment characteristics |
| 8 | Total Cost of Ownership (TCO) model |
| 9 | Scaling sensitivity |
| 10 | Decision matrix & recommendation |
| 11 | Save your work |

<div style="background-color:#FCF3D9; border-left:5px solid #B58800; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B58800;">Setup</b><br>Same shared project as Week 1. First time? Run <code>00_Setup.ipynb</code> in the project root first. Otherwise just select the <code>.venv</code> kernel and run the cell below.</div>

In [ ]:
import sys
from pathlib import Path

def find_project_root():
    '''Walk up from the current working directory until we find the
    Course_Project root (identified by having both requirements.txt
    and a src/ folder). Works no matter which VS Code cwd setting is in
    effect for notebooks (fileDirname or workspaceFolder), because either
    way this notebook lives somewhere inside the project tree.'''
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the Course_Project root (a folder containing both "
        "requirements.txt and src/). Open this notebook from inside the project "
        "folder in VS Code (File > Open Folder > Course_Project), or set PROJECT_ROOT "
        "manually, e.g. PROJECT_ROOT = Path(r'C:\path\to\INFT41000\Course_Project')."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"

for required_dir in (DATA_DIR, MODELS_DIR):
    if not required_dir.is_dir():
        raise FileNotFoundError(
            f"Missing folder: {required_dir}\n"
            "Create it first (see the folder-structure instructions in 00_Setup.ipynb) -- "
            "this notebook does not create folders for you."
        )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import math
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Consistent colors for On-Prem / Cloud / Hybrid across every chart later this week
ENV_COLORS = {"On-Prem": "#5A6066", "Cloud": "#005291", "Hybrid": "#B58800"}
print("Setup OK")

## 1. Load Week 1's Artifacts

In [ ]:
DATASET_PATH = DATA_DIR / "fraudguard_transactions_week1.csv"
MODEL_PATH = MODELS_DIR / "week1_baseline_model.joblib"

for p in (DATASET_PATH, MODEL_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {p.name} in {p.parent}. Run Week 1's notebook first "
            "(the \"Save Your Work\" section) -- this notebook does not regenerate it for you."
        )

df = pd.read_csv(DATASET_PATH)
clf = joblib.load(MODEL_PATH)
print(f"Loaded dataset: {df.shape}, model: {type(clf).__name__}")

## 2. The Readiness Framework

Seven dimensions, each scored **1 (not ready) to 5 (fully ready)**, combined into one weighted
score:

| Dimension | Weight | What it asks |
|---|---|---|
| Data Readiness | 20% | Is the data complete, clean, and representative? |
| Model Readiness | 20% | Does the model perform reliably, including on unseen data? |
| Infrastructure Readiness | 15% | Is there a deployment environment chosen and ready? |
| Organizational Readiness | 10% | Does the team/org have the skills and process to run this? |
| Risk | 15% | Legal, safety, fairness, reputational exposure if this fails? |
| Cost | 10% | Is the ongoing cost known and approved? |
| Operational Readiness | 10% | Monitoring, support, incident response in place? |

<div style="background-color:#FDE8E6; border-left:5px solid #B22222; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B22222;">The hard-blocker rule</b><br>A weighted average can hide a single critical failure — same lesson as Week 1's accuracy paradox. So: <b>any dimension scoring 1 forces an automatic No-Go</b>, no matter how high the weighted average is. A great model with an unresolved legal blocker still doesn't ship.</div>

## 3. Data Readiness

### Exercise 2.1
Write `data_readiness_report(df)`, returning a dict with: `n_rows`, `missing_pct` (% of all
cells that are null), `duplicate_pct` (% of fully duplicate rows), `fraud_rate_pct`, and
`n_transaction_types`. Print it, then in the markdown cell below assign a **Data Readiness
score (1–5)** with a one-sentence justification referencing your actual numbers.

In [ ]:
# YOUR CODE HERE
def data_readiness_report(df):
    raise NotImplementedError

report = data_readiness_report(df)
report

*Data Readiness score (1–5): ___ — Justification:*

<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">A clean report isn't the whole story</b><br>This synthetic set is unusually clean. Real production data readiness also checks referential integrity across systems, drift vs. the training distribution, PII exposure, and timeliness — Week 4 (data pipelines) goes deeper here.</div>

## 4. Model Readiness

Testing on the same train/test split as Week 1 isn't a fair readiness check — the model has
effectively "seen" that data distribution already. A better test: **freshly generated,
never-touched data** (a different random seed — same underlying patterns, new draws).

### Exercise 2.2
Generate a fresh dataset with `seed=123` (same `n_rows`, `fraud_rate`, `missing_rate` as Week
1), evaluate the loaded model on it, and print a classification report. Then assign a **Model
Readiness score (1–5)** with justification below.

In [ ]:
# YOUR CODE HERE
from fraudguard_datagen import generate_transactions

features = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
raise NotImplementedError

*Model Readiness score (1–5): ___ — Justification:*

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">Scenario briefing</b><br>FraudGuard is being proposed for a pilot at a mid-size payments company. IT has general cloud experience but has never run an ML model in production. Compliance requires that any blocked transaction come with an explainable reason. Finance has approved a 3-month pilot budget only, not a full production budget.</div>

## 5. Infrastructure, Organizational, Risk, Cost & Operational Readiness

These five can't be computed from code yet — no deployment environment is chosen until later
in this same notebook (Section 10), and no monitoring/ops exists until Week 9. Score them
from the scenario above and what you know about FraudGuard's current state (a notebook
prototype, per Week 1).

### Exercise 2.3
Fill in a score (1–5) and one-sentence justification for each row:

| Dimension | Score (1–5) | Justification |
|---|---|---|
| Infrastructure Readiness | | |
| Organizational Readiness | | |
| Risk | | |
| Cost | | |
| Operational Readiness | | |

## 6. Build the Scorecard & Recommendation

### Exercise 2.4
Write `assess_readiness(scores, weights)` that:
1. Builds a DataFrame with columns `dimension`, `score`, `weight`, `weighted` (`score * weight`).
2. Computes `overall = weighted.sum()`.
3. Applies the hard-blocker rule: if **any** score is `1`, recommendation is `"NO-GO (hard blocker)"`.
4. Otherwise: `"GO"` if `overall >= 4.0`, `"CONDITIONAL GO"` if `overall >= 2.5`, else `"NO-GO"`.
5. Returns `(scorecard_df, overall, recommendation)`.

Then plug in your own scores from Exercises 2.1–2.3 below and run it.

In [ ]:
# YOUR CODE HERE
WEIGHTS = {
    "Data Readiness": 0.20,
    "Model Readiness": 0.20,
    "Infrastructure Readiness": 0.15,
    "Organizational Readiness": 0.10,
    "Risk": 0.15,
    "Cost": 0.10,
    "Operational Readiness": 0.10,
}

def assess_readiness(scores, weights):
    raise NotImplementedError

# Fill in with YOUR scores from Exercises 2.1-2.3
my_scores = {
    "Data Readiness": None,
    "Model Readiness": None,
    "Infrastructure Readiness": None,
    "Organizational Readiness": None,
    "Risk": None,
    "Cost": None,
    "Operational Readiness": None,
}

scorecard_df, overall_score, recommendation = assess_readiness(my_scores, WEIGHTS)
print(scorecard_df)
print(f"\nOverall weighted score: {overall_score:.2f}/5")
print(f"Recommendation: {recommendation}")

*In 3–4 sentences: do you agree with your own recommendation? What's the single gap you'd fix first, and which future week of this course addresses it?*

<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">Picking up the Infrastructure gap</b><br>Section 5 almost certainly scored Infrastructure Readiness low — no deployment environment has been chosen yet. The rest of this week fixes exactly that: compare on-premise, cloud, and hybrid deployment for FraudGuard using a real cost model and a weighted decision matrix, and make (and justify) an actual recommendation.</div>

## 7. Environment Characteristics

| | On-Prem | Cloud (AWS) | Hybrid |
|---|---|---|---|
| Cost model | High upfront capex, fixed capacity | Pay-as-you-go opex, scales with usage | Mixed capex + opex |
| Control | Full control over hardware & data | Shared responsibility with provider | Sensitive parts controlled, rest shared |
| Performance | Predictable, low network latency to on-site systems | Depends on region/instance choice, can autoscale | Best-of-both if designed well, added network hops |
| Security | You own the entire stack (and every patch) | Provider secures the infrastructure layer, you secure your app/data (Week 6) | Two security boundaries to manage, not one |
| Scalability | Capacity-limited by hardware you own | Near-instant elastic scaling (Week 8) | Cloud portion scales; on-prem portion doesn't |
| Compliance / data residency | Easiest to guarantee data never leaves the building | Depends on provider region & certifications | Sensitive data can stay on-prem by design |
| Time-to-deploy | Weeks to months (procurement, install) | Hours to days | Slower than pure cloud (two environments to stand up) |

<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">These numbers are illustrative</b><br>The cost model below uses made-up (but realistic-shaped) unit costs to teach the <b>method</b> of TCO modeling — not real AWS quotes. For actual pricing, use the real <a href="https://calculator.aws">AWS Pricing Calculator</a>. We revisit real cost tracking with AWS Cost Explorer in Week 8.</div>

## 8. Total Cost of Ownership (TCO) Model

### Exercise 2.5
Implement all three functions below for **3-year TCO**, following this spec exactly:

**On-Prem** — servers handle 500,000 txn/day each (round up to whole servers):
`hardware_capex = $80,000 × n_servers` (one-time) · `annual_hosting = $12,000 × n_servers`
· `annual_staff = $55,000 + $15,000 × (n_servers − 1)`

**Cloud** — always-on compute + usage-based pricing:
`annual_compute = $0.20/hr × 24 × 365` · `annual_variable = (txn_per_day × 365 / 1000) × $0.12`
· `annual_storage = $600` · `annual_staff = $25,000`

**Hybrid** — smaller on-prem footprint (800,000 txn/day/server) + cloud overflow + integration:
`hardware_capex = $35,000 × n_servers` · `annual_hosting = $6,000 × n_servers`
· `annual_staff = $45,000` · `annual_cloud_compute = $0.20/hr × 24 × 365`
· `annual_variable = (txn_per_day × 365 / 1000) × $0.06` · `annual_integration = $8,000`

3-year TCO = one-time costs + 3 × (sum of that environment's annual costs).

In [ ]:
# YOUR CODE HERE -- implement all three following the spec above
def tco_on_prem(txn_per_day, years=3):
    raise NotImplementedError

def tco_cloud(txn_per_day, years=3):
    raise NotImplementedError

def tco_hybrid(txn_per_day, years=3):
    raise NotImplementedError

PILOT_VOLUME = 50_000  # txn/day, matches this week's pilot scenario

tco_pilot = {
    "On-Prem": tco_on_prem(PILOT_VOLUME),
    "Cloud": tco_cloud(PILOT_VOLUME),
    "Hybrid": tco_hybrid(PILOT_VOLUME),
}
for env, cost in tco_pilot.items():
    print(f"{env:8s}  3-yr TCO at {PILOT_VOLUME:,} txn/day: ${cost:,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
envs = list(tco_pilot.keys())
values = list(tco_pilot.values())
bars = ax.bar(envs, values, color=[ENV_COLORS[e] for e in envs], width=0.5)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v, f"${v:,.0f}", ha="center", va="bottom")
ax.set_ylabel("3-year TCO ($)")
ax.set_title(f"3-Year TCO at {PILOT_VOLUME:,} transactions/day (pilot volume)")
plt.tight_layout()
plt.show()

## 9. Scaling Sensitivity

FraudGuard won't stay at pilot volume forever (Week 8 covers scaling in depth). Does the
ranking above hold as transaction volume grows?

In [ ]:
volumes = [10_000, 50_000, 200_000, 500_000, 1_000_000, 2_000_000, 5_000_000, 10_000_000]

sensitivity = pd.DataFrame({
    "txn_per_day": volumes,
    "On-Prem": [tco_on_prem(v) for v in volumes],
    "Cloud": [tco_cloud(v) for v in volumes],
    "Hybrid": [tco_hybrid(v) for v in volumes],
})

fig, ax = plt.subplots(figsize=(8, 4.5))
for env in ["On-Prem", "Cloud", "Hybrid"]:
    ax.plot(sensitivity["txn_per_day"], sensitivity[env], marker="o", linewidth=2,
            color=ENV_COLORS[env], label=env)
ax.set_xscale("log")
ax.set_xlabel("Transactions/day (log scale)")
ax.set_ylabel("3-year TCO ($)")
ax.set_title("TCO vs. Transaction Volume")
ax.legend()
plt.tight_layout()
plt.show()

sensitivity

*In this model, does any environment ever become cheaper than Cloud, even at 10M txn/day? Why might that not hold in the real world at truly massive, sustained scale (hint: Dropbox famously migrated much of its storage OFF the cloud and built its own infrastructure — "Magic Pocket" — saving roughly $75M over two years once its scale was large enough to justify it)?*

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">Scenario reminder (from Section 5 above)</b><br>FraudGuard is a pilot at a mid-size payments company. IT has general cloud experience but has never run an ML model in production. Compliance requires explainable decisions for every blocked transaction. Finance approved a 3-month pilot budget only.</div>

## 10. Decision Matrix & Recommendation

Cost is only one criterion — Section 7's characteristics matter too. Score each environment
1 (worst) – 5 (best) on the six criteria below, using the scenario and Section 7's table.
**Cost is scored for you**, computed from Section 8's numbers (cheapest = 5).

### Exercise 2.6
Fill in your own scores (1–5) for the other six criteria:

| Criterion | Weight | On-Prem | Cloud | Hybrid |
|---|---|---|---|---|
| Cost *(auto-scored below)* | 25% | — | — | — |
| Control | 15% | | | |
| Performance / Latency | 15% | | | |
| Security | 15% | | | |
| Scalability | 15% | | | |
| Compliance / Data Residency | 10% | | | |
| Time-to-Deploy | 5% | | | |

### Exercise 2.7
Write `rank_environments(cost_tco, qualitative_scores, weights)` that:
1. Converts each environment's TCO into a Cost score 1–5 (cheapest of the three = 5, most
   expensive = 1, linearly scaled in between — cheapest and most expensive never tie).
2. Builds one DataFrame per environment combining Cost with your six qualitative scores.
3. Computes each environment's weighted total.
4. Returns a DataFrame ranked highest score first.

In [ ]:
# YOUR CODE HERE
WEIGHTS = {
    "Cost": 0.25, "Control": 0.15, "Performance": 0.15, "Security": 0.15,
    "Scalability": 0.15, "Compliance": 0.10, "Time-to-Deploy": 0.05,
}

def cost_to_score(tco_dict):
    raise NotImplementedError

def rank_environments(cost_tco, qualitative_scores, weights):
    raise NotImplementedError

# Fill in with YOUR scores from Exercise 2.6
qualitative_scores = {
    "On-Prem": {"Control": None, "Performance": None, "Security": None, "Scalability": None, "Compliance": None, "Time-to-Deploy": None},
    "Cloud":   {"Control": None, "Performance": None, "Security": None, "Scalability": None, "Compliance": None, "Time-to-Deploy": None},
    "Hybrid":  {"Control": None, "Performance": None, "Security": None, "Scalability": None, "Compliance": None, "Time-to-Deploy": None},
}

ranking = rank_environments(tco_pilot, qualitative_scores, WEIGHTS)
ranking

*Does the top-ranked environment match the cheapest one from Section 8? If not, what non-cost criterion changed the outcome? Write 3–4 sentences justifying FraudGuard's final environment recommendation.*

## 11. Save Your Work — Needed for Week 3

In [ ]:
recommended_env = ranking.iloc[0]["environment"]

scorecard_df.to_csv(DATA_DIR / "readiness_scorecard.csv", index=False)
with open(DATA_DIR / "readiness_recommendation.txt", "w") as f:
    f.write(f"Overall score: {overall_score:.2f}/5\nRecommendation: {recommendation}\n")

ranking.to_csv(DATA_DIR / "environment_decision_matrix.csv", index=False)
with open(DATA_DIR / "environment_recommendation.txt", "w") as f:
    f.write(f"Recommended environment: {recommended_env}\n")
    f.write(f"Weighted score: {ranking.iloc[0]['weighted_total']}/5\n")

print(f"Recommended environment: {recommended_env}")
print(f"Saved: {(DATA_DIR / 'readiness_scorecard.csv').resolve()}")
print(f"Saved: {(DATA_DIR / 'readiness_recommendation.txt').resolve()}")
print(f"Saved: {(DATA_DIR / 'environment_decision_matrix.csv').resolve()}")
print(f"Saved: {(DATA_DIR / 'environment_recommendation.txt').resolve()}")

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">Submit</b><br>This notebook, run top-to-bottom with no errors · Exercises 2.1–2.4 completed with real readiness scores and justifications · Exercises 2.5–2.7 completed with your own qualitative scores and justification · both reflection questions answered.<br><br><b>Next week:</b> Week 3 — standing up real cloud infrastructure (AWS: S3, IAM, compute) for whichever environment you recommended, then drawing FraudGuard's reference architecture. Even an on-prem or hybrid recommendation still needs cloud skills — almost every real deployment includes one.</div>